# 🏗️ Construction Site Safety Monitoring — Training Notebook

Complete training pipeline for YOLOv8 construction safety detection model.

**Model**: YOLOv8-Medium (balance of speed and accuracy)
**Classes**: worker, helmet, no_helmet, vest, no_vest, harness, machinery, danger_zone
**Target**: mAP@0.5 ≥ 0.72 | Inference ≥ 15 FPS on T4 GPU

## Cell Group 1 — Environment Setup

In [ ]:
# Install dependencies
!pip install -q ultralytics roboflow supervision albumentations gdown

# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('⚠️ No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

## Cell Group 2 — Dataset Download & Validation

In [ ]:
# Option A: Download from Roboflow (recommended)
# Replace with your actual Roboflow API key and project
from roboflow import Roboflow

# 📌 REQUIRES: Replace with your Roboflow API key
# rf = Roboflow(api_key='YOUR_API_KEY')
# project = rf.workspace('YOUR_WORKSPACE').project('construction-safety')
# dataset = project.version(1).download('yolov8', location='/content/construction_safety')

# Option B: Download from Google Drive
# !gdown 'YOUR_GDRIVE_FILE_ID' -O /content/construction_safety.zip
# !unzip -q /content/construction_safety.zip -d /content/construction_safety

print('📌 Uncomment one of the download options above and add your credentials.')
print('Expected structure:')
print('  /content/construction_safety/')
print('  ├── images/{train,val,test}/')
print('  └── labels/{train,val,test}/')

In [ ]:
# Validate dataset structure
import os
from pathlib import Path

DATA_ROOT = Path('/content/construction_safety')
for split in ['train', 'val', 'test']:
    img_dir = DATA_ROOT / 'images' / split
    lbl_dir = DATA_ROOT / 'labels' / split
    if img_dir.exists():
        imgs = len(list(img_dir.glob('*.*')))
        lbls = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
        print(f'  {split}: {imgs} images, {lbls} labels')
    else:
        print(f'  {split}: ❌ not found')

## Cell Group 3 — Augmentation Pipeline

In [ ]:
import albumentations as A
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Construction-site-specific augmentation pipeline
train_transform = A.Compose([
    A.RandomResizedCrop(size=(640, 640), scale=(0.5, 1.0), ratio=(0.8, 1.2), p=0.5),
    A.Resize(640, 640),
    A.HorizontalFlip(p=0.5),
    A.Perspective(scale=(0.02, 0.08), p=0.3),
    # Lighting variation — harsh sun, shadows, overcast
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
    # HSV shift — hi-vis vest color robustness across lighting
    A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=40, val_shift_limit=40, p=0.5),
    # CLAHE — simulates low-light preprocessing
    A.CLAHE(clip_limit=4.0, p=0.2),
    # Motion blur — camera/worker movement, machinery vibration
    A.MotionBlur(blur_limit=7, p=0.25),
    # Gaussian noise — surveillance camera sensor noise
    A.GaussNoise(std_range=(0.02, 0.08), p=0.2),
    # Random occlusion — scaffold poles, equipment blocking view
    A.CoarseDropout(num_holes_range=(1, 3), hole_height_range=(12, 50),
                    hole_width_range=(12, 50), fill='random', p=0.3),
    # Weather: fog + rain
    A.RandomFog(fog_coef_range=(0.1, 0.3), p=0.1),
    A.RandomRain(slant_range=(-10, 10), drop_length=15, p=0.1),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'],
                            min_area=100, min_visibility=0.3))

print('✅ Augmentation pipeline ready')
print(f'   {len(train_transform.transforms)} transforms configured')

## Cell Group 4 — YOLOv8 Training

In [ ]:
# Write dataset YAML config
yaml_content = """
path: /content/construction_safety
train: images/train
val: images/val
test: images/test
nc: 8
names:
  0: worker
  1: helmet
  2: no_helmet
  3: vest
  4: no_vest
  5: harness
  6: machinery
  7: danger_zone
"""
with open('/content/construction_safety.yaml', 'w') as f:
    f.write(yaml_content)
print('✅ Dataset YAML written')

In [ ]:
from ultralytics import YOLO

# Load YOLOv8-Medium pretrained on COCO
model = YOLO('yolov8m.pt')

# Train with construction-site-optimized hyperparameters
results = model.train(
    data='/content/construction_safety.yaml',
    epochs=100,       # Sufficient for simple PPE shapes
    imgsz=640,        # Standard for surveillance resolution
    batch=16,         # Max for T4 16GB with YOLOv8m
    lr0=0.01,         # Standard SGD LR for COCO pretrained
    lrf=0.01,         # Cosine decay to 1% of lr0
    momentum=0.937,   # Stable convergence with class imbalance
    weight_decay=0.0005,  # Mild L2 for small dataset
    warmup_epochs=3,  # Gradual warmup for domain transfer
    patience=20,      # Allow late-stage improvements
    device=0,
    project='safety_monitor',
    name='yolov8m_construction_v1',
    exist_ok=True,
    val=True,
    plots=True,
    save_period=10,   # Checkpoint every 10 for Colab stability
)

## Cell Group 5 — Training Curves & Evaluation

In [ ]:
from IPython.display import Image, display
import glob

# Display training curves
results_dir = 'safety_monitor/yolov8m_construction_v1'
for plot in ['results.png', 'confusion_matrix.png',
             'PR_curve.png', 'F1_curve.png']:
    path = f'{results_dir}/{plot}'
    if os.path.exists(path):
        print(f'\n📊 {plot}')
        display(Image(filename=path, width=800))

In [ ]:
# Evaluate on test set
model = YOLO('safety_monitor/yolov8m_construction_v1/weights/best.pt')
metrics = model.val(
    data='/content/construction_safety.yaml',
    split='test',
    device=0,
    plots=True,
)

print(f'\n📊 Test Set Results:')
print(f'  mAP@0.5:      {metrics.box.map50:.4f}')
print(f'  mAP@0.5:0.95: {metrics.box.map:.4f}')
print(f'\n  Per-class AP@0.5:')
for i, name in model.names.items():
    if i < len(metrics.box.ap50):
        print(f'    {name:<15} {metrics.box.ap50[i]:.4f}')

## Cell Group 6 — Secondary Attribute Classifier (Optional)

In [ ]:
# Fine-tune MobileNetV3 on cropped worker images for attribute classification
# Classes: helmet_correct / helmet_missing / helmet_incorrect
#          vest_present / vest_absent / vest_open

import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset

class PPEAttributeClassifier(nn.Module):
    """MobileNetV3-Small classifier for PPE attribute verification."""
    def __init__(self, num_classes: int = 6):
        super().__init__()
        self.backbone = models.mobilenet_v3_small(weights='DEFAULT')
        in_features = self.backbone.classifier[-1].in_features
        self.backbone.classifier[-1] = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.backbone(x)

# Define transforms for cropped worker images
attribute_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

ATTRIBUTE_CLASSES = [
    'helmet_correct', 'helmet_missing', 'helmet_incorrect',
    'vest_present', 'vest_absent', 'vest_open',
]

print(f'Model params: {sum(p.numel() for p in PPEAttributeClassifier().parameters()):,}')
print('📌 REQUIRES: Cropped worker images labeled with attribute classes.')
print('   Use YOLO detections to crop workers, then manually label attributes.')

In [ ]:
# Save best model to Google Drive
import shutil

drive_path = '/content/drive/MyDrive/construction_safety_models/'
os.makedirs(drive_path, exist_ok=True)
shutil.copy('safety_monitor/yolov8m_construction_v1/weights/best.pt',
            drive_path + 'best.pt')
print(f'✅ Best model saved to Google Drive: {drive_path}best.pt')